# Migration discourse pilot - France 2018

In [ ]:
import sys
from pathlib import Path

# Make the src package importable from the notebook
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
import matplotlib.pyplot as plt

from src import load, filters, typology, inspect
from src.config import PROCESSED_DIR, SOURCE_COUNTRY, SOURCE_YEAR

In [ ]:
lf = load.load_facts_lazy()
load.inspect_schema(lf)

In [ ]:
inspect.show_topic_distribution(lf, top_n=15)

In [ ]:
mentions = filters.build_migration_mentions(lf)
print(f"Foreign country mentions in {SOURCE_COUNTRY} {SOURCE_YEAR} migration debates: "
      f"{mentions.height:,}")
mentions.head(10)

In [ ]:
top = inspect.show_top_countries(mentions, top_n=20)
top

In [ ]:
annotated = typology.apply_typology(mentions)
annotated.select(["entity_content", "ref_type", "sentiment_bucket",
                  "sentence_sentiment_value"]).head(10)

In [ ]:
matrix = typology.build_2x3_matrix(annotated)
matrix

In [ ]:
top_5_countries = top.head(5)["entity_content"].to_list()
country_matrix = typology.matrix_by_country(
    annotated.filter(pl.col("entity_content").is_in(top_5_countries)),
    min_mentions=3,
)
country_matrix

In [ ]:
# Look at Germany mentions - read 5 random ones and check whether the
# heuristic classification matches your reading
inspect.show_sample_contexts(annotated, country="Allemagne", n_samples=5)

In [ ]:
out_path = PROCESSED_DIR / f"{SOURCE_COUNTRY}_{SOURCE_YEAR}_migration_mentions.parquet"
annotated.write_parquet(out_path)
print(f"Saved {annotated.height:,} rows to {out_path}")

In [ ]:
matrix_pd = matrix.to_pandas().set_index("ref_type")

fig, ax = plt.subplots(figsize=(8, 4))
matrix_pd.plot(kind="bar", stacked=False, ax=ax,
               color=["#1d9e75", "#a32d2d", "#888780"])
ax.set_title(f"Migration discourse 2x3 typology - {SOURCE_COUNTRY} {SOURCE_YEAR}")
ax.set_xlabel("Reference type")
ax.set_ylabel("Number of mentions")
ax.legend(title="Sentiment")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()